test cell

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import io

#knn alg
def knn_predict_one(x, X_train, y_train, k):
    dists = np.sqrt(np.sum((X_train - x) ** 2, axis=1))
    k_idx = np.argsort(dists)[:k]
    
    labels, counts = np.unique(y_train[k_idx], return_counts=True)
    return labels[np.argmax(counts)]

#process german.data / train model
try:
    #columns 0 checking, 1 duration, 2 history, 3 purpose, 4:amount, 5 savings, 6 employ, 8 sex, 11 property, 14 housing
    data_columns = [0, 1, 2, 3, 4, 5, 6, 8, 11, 14]
    
    #load csv
    data = pd.read_csv("german.data", header=None, delim_whitespace=True)
    
    x_axis = data.iloc[:, data_columns].copy()
    y = data.iloc[:, -1].values  # 1 = Good, 2 = Bad

    for i, column_index in enumerate(x_axis.columns):
        if x_axis[column_index].dtype == "object":
            x_axis[column_index] = x_axis[column_index].astype("category")
            x_axis[column_index] = x_axis[column_index].cat.codes

    X = x_axis.values.astype(float)
    
    #split data
    np.random.seed(42)
    indices = np.random.permutation(len(X))
    split = int(len(X) * 0.7)
    training_index, test_index = indices[:split], indices[split:]
    x_training, y_training = X[training_index], y[training_index]
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Make sure 'german.data' is in the same folder as this notebook.")
    x_training = [] 

# ui widgets

style = {'description_width': '150px'}
layout = widgets.Layout(width='400px', margin='5px')

header = widgets.HTML("<h2> German Credit Risk System</h2>")
desc = widgets.HTML("Adjust the inputs and click <b>'Predict Risk'</b>. Then check the <b>Data Insights</b> tab to see the graphs update.")

# info input
w_checking = widgets.Dropdown(options=[('< 0 DM', 0), ('0 <= ... < 200 DM', 1), ('>= 200 DM', 2), ('No chequing account', 3)], description='Chequing Status:', style=style, layout=layout)
w_duration = widgets.IntSlider(value=12, min=4, max=72, description='Duration (Months):', style=style, layout=layout)
w_history = widgets.Dropdown(options=[('No credits/All paid', 0), ('All paid back', 1), ('Existing paid', 2), ('Delay in past', 3), ('Critical account', 4)], description='Credit History:', style=style, layout=layout)
w_purpose = widgets.Dropdown(options=[('Car (New)', 0), ('Car (Used)', 1), ('Furniture', 2), ('Radio/TV', 3), ('Appliances', 4), ('Repairs', 5), ('Education', 6), ('Retraining', 8), ('Business', 9), ('Other', 10)], description='Purpose:', style=style, layout=layout)
w_amount = widgets.IntText(value=1500, description='Credit Amount:', style=style, layout=layout)
w_savings = widgets.Dropdown(options=[('< 100 DM', 0), ('100 <= ... < 500', 1), ('500 <= ... < 1000', 2), ('>= 1000 DM', 3), ('Unknown/No Savings', 4)], description='Savings:', style=style, layout=layout)
w_employ = widgets.Dropdown(options=[('Unemployed', 0), ('< 1 year', 1), ('1 <= ... < 4 yrs', 2), ('4 <= ... < 7 yrs', 3), ('>= 7 years', 4)], description='Employment:', style=style, layout=layout)
w_sex = widgets.Dropdown(options=[('Male: Divorced', 0), ('Female: Div/Mar', 1), ('Male: Single', 2), ('Male: Mar/Wid', 3)], description='Sex & Status:', style=style, layout=layout)
w_property = widgets.Dropdown(options=[('Real Estate', 0), ('Savings Agreement', 1), ('Car / Other', 2), ('Unknown / No Property', 3)], description='Property:', style=style, layout=layout)
w_housing = widgets.Dropdown(options=[('Rent', 0), ('Own', 1), ('For Free', 2)], description='Housing:', style=style, layout=layout)

btn_predict = widgets.Button(description="Predict Risk", button_style='primary', icon='check', layout=widgets.Layout(width='200px'))
out_predict = widgets.Output()
output_visibility = widgets.Output()

# visualize graphs
def show_visuals(highlight_duration=None, highlight_amount=None, predicted_class=None):
    with output_visibility:
        clear_output(wait=True)
        if len(x_training) == 0: return
        
        fig, (axis1, axis2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # graph 1: bar graph
        unique, counts = np.unique(y, return_counts=True)
        bar_colors = ['green' if u == 1 else 'red' for u in unique]
        if predicted_class is not None:
            index = 0 if predicted_class == 1 else 1
            bar_colors[index] = 'blue' # highlighted result
            
        axis1.bar(['Good (1)', 'Bad (2)'], counts, color=bar_colors, alpha=0.7)
        axis1.set_ylabel("# of people from dataset")
        axis1.set_title("Risk Distribution (Blue = Your Prediction)")
        
        # graph 2: scatter plot
        subset = data.iloc[:300]
        colors_scatter = ['green' if c == 1 else 'red' for c in subset.iloc[:, -1]]
        axis2.scatter(subset[1], subset[4], c=colors_scatter, alpha=0.3, label='Past Data')
        
        # user inputed data location star
        if highlight_duration is not None:
            axis2.scatter(highlight_duration, highlight_amount, s=300, c='blue', marker='*', edgecolors='black', label='You')
            
        axis2.set_xlabel("Duration")
        axis2.set_ylabel("Amount")
        axis2.set_title("Where do you fit?")
        axis2.legend()
        axis2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

# button click function
def on_predict_click(b):
    with out_predict:
        clear_output()
        inputs = [
            w_checking.value, w_duration.value, w_history.value, w_purpose.value,
            w_amount.value, w_savings.value, w_employ.value, w_sex.value,
            w_property.value, w_housing.value
        ]
        input_array = np.array([inputs], dtype=float)
        
        # make prediction
        prediction = knn_predict_one(input_array, x_training, y_training, k=5)
        
        if prediction == 1:
            display(widgets.HTML(f"<h3 style='color:green; border:2px solid green; padding:10px;'> Prediction: Good Credit Risk</h3>"))
        else:
            display(widgets.HTML(f"<h3 style='color:red; border:2px solid red; padding:10px;'> Prediction: Bad Credit Risk</h3>"))
            
        # Update Graphs
        show_visuals(w_duration.value, w_amount.value, prediction)

btn_predict.on_click(on_predict_click)

# create notebook layout
input_column1 = widgets.VBox([w_checking, w_duration, w_history, w_purpose, w_amount])
input_column2 = widgets.VBox([w_savings, w_employ, w_sex, w_property, w_housing])
input_ui = widgets.VBox([
    widgets.HBox([input_column1, input_column2]),
    widgets.HTML("<br>"),
    btn_predict,
    out_predict,
    output_visibility
])

# intialize empty graphs
show_visuals()


C:\Users\lerya.DESKTOP-O4134LB\AppData\Local\Temp\ipykernel_10700\4198390292.py:22: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv("german.data", header=None, delim_whitespace=True)


In [45]:
# importance of each variable

feature_names = [
    'Checking Status',
    'Duration (Months)',
    'Credit History',
    'Purpose',
    'Credit Amount',
    'Savings',
    'Employment',
    'Sex & Status',
    'Property',
    'Housing'
]

def calculate_feature_contributions(x, X_train, y_train, k=5):
    #calculate how much each input feature contributes to the prediction
    x = np.array(x).flatten()
    
    #calculate distances
    dists = np.sqrt(np.sum((X_train - x) ** 2, axis=1))
    k_index = np.argsort(dists)[:k]
    
    #get the k nearest neighbors
    neighbors = X_train[k_index]
    neighbor_distances = dists[k_index]
    
    #calculate feature-wise contribution (normalized distance per feature)
    #features with larger differences contribute more
    feature_diffs = np.abs(neighbors - x)
    
    #weight by inverse distance (closer neighbors matter more)
    weights = 1.0 / (neighbor_distances + 1e-9)
    weights = weights / np.sum(weights)  #normalize
    
    #calculate weighted contribution per feature
    contributions = np.zeros(len(feature_names))
    for j in range(len(feature_names)):
        contributions[j] = np.sum(weights * feature_diffs[:, j])
    
    #normalize contributions to 0-100
    if np.sum(contributions) > 0:
        contributions = (contributions / np.sum(contributions)) * 100
    else:
        contributions = np.ones(len(feature_names)) * (100 / len(feature_names))
    
    return contributions

out_importance = widgets.Output()

def on_importance_click(b):
    with out_importance:
        clear_output()
        if len(x_training) == 0:
            print("Error: Model not trained.")
            return
        
        #grab current input
        inputs = np.array([
            w_checking.value, w_duration.value, w_history.value, w_purpose.value,
            w_amount.value, w_savings.value, w_employ.value, w_sex.value,
            w_property.value, w_housing.value
        ], dtype=float)
        
        #calculate feature contributions
        contributions = calculate_feature_contributions(inputs, x_training, y_training, k=5)
        
        #create bar graph
        fig, axis = plt.subplots(figsize=(12, 6))
        
        colors = plt.cm.RdYlGn_r(contributions / 100)  # Red = high importance, Green = low
        bars = axis.barh(feature_names, contributions, color=colors)
        
        axis.set_xlabel('Contribution to Risk Calculation (%)', fontsize=12)
        axis.set_title('Feature Importance: How Each Input Affects Your Risk Score', fontsize=14, fontweight='bold')
        axis.set_xlim(0, 100)
        
        #add bar value labels
        for i, (bar, val) in enumerate(zip(bars, contributions)):
            axis.text(val + 1, i, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        #show breakdown
        display(widgets.HTML("<h3> Feature Importance Breakdown</h3>"))

btn_importance = widgets.Button(description="Calculate Importance", button_style='info', layout=widgets.Layout(width='200px'))
btn_importance.on_click(on_importance_click)

importance_ui = widgets.VBox([
    widgets.HTML("<h2> Feature Importance Analysis</h2>"),
    widgets.HTML("<p>Click the button below to see how much each input contributes to your risk prediction. Features with higher percentages have more influence on the final score.</p>"),
    btn_importance,
    out_importance
])


In [46]:
# ==========================================
# 6. UPDATE UI WITH ALL TABS
# ==========================================

# Update the tabs to include importance tab
tabs = widgets.Tab([input_ui, importance_ui])
tabs.set_title(0, 'Risk Predictor')
tabs.set_title(1, 'Feature Importance')

# Update final UI
ui = widgets.VBox([header, desc, tabs])

# Re-display the updated UI
display(ui)